# Fakultätszuordnung mit OpenAlex und GERiT

1. Schritt: Installieren der erforderlichen Python-Pakete:

In [1]:
import sys
#!{sys.executable} -m pip install pyalex
#!{sys.executable} -m pip install requests

In [2]:
import pyalex
import requests
import re

2. Schritt: OpenAlex API Key einfügen:

In [9]:
OPENALEX_API_KEY = 'OPENALEX_API_KEY'
pyalex.config.api_key = OPENALEX_API_KEY

3. Schritt: Filter anpassen

In [4]:
institution_ids = ['https://openalex.org/I74656192', # GAU
                   'https://openalex.org/I4387154063', # SUB
                   'https://openalex.org/I4210091733', # GWDG
                   'https://openalex.org/I4210116730' # UMG
                  ]

item_types = ['article', 'review']

publication_years = ['2025']

sample_size = 10

4. Schritt: Mapping herunterladen 

In [5]:
gerit_gau_url = 'https://raw.githubusercontent.com/naustica/lower_saxony_institutions/main/data/mappings/gau.json'
response = requests.get(gerit_gau_url)
gerit_mapping = response.json()

In [6]:
def map_address(address: str, mapping_dict: dict):
    faculty = None
    faculty_id = None
    department = None
    department_id = None
    institute = None
    institute_id = None
    
    for k, v in mapping_dict.items():
        pattern = re.compile(r'|'.join(v.get('faculty_patterns')), re.IGNORECASE)
        res = bool(pattern.search(address))
        if res:
            faculty = k
            faculty_id = v.get('faculty_id')

    if faculty:
        for k, v in mapping_dict.get(faculty).get('suborganisations').items():
            pattern = re.compile(r'|'.join(v.get('department_patterns')), re.IGNORECASE)
            res = bool(pattern.search(address))
            if res:
                department = k
                department_id = v.get('department_id')

    if department:
        for k, v in mapping_dict.get(faculty).get('suborganisations').get(department).get('suborganisations').items():
            pattern = re.compile(r'|'.join(v.get('institute_patterns')), re.IGNORECASE)
            res = bool(pattern.search(address))
            if res:
                institute = k
                institute_id = v.get('institute_id')

    return dict(faculty=faculty, 
                faculty_id=faculty_id,
                department=department, 
                department_id=department_id, 
                institute=institute,
                institute_id=institute_id)

5. Schritt: OpenAlex API Anfrage erstellen

In [7]:
response = pyalex.Works().filter(
    authorships={'affiliations': {'institution_ids': '|'.join(institution_ids)}}, 
    type='|'.join(item_types), 
    publication_year='|'.join(publication_years)).sample(sample_size).get()

6. Schritt: API-Antwort auswerten

In [8]:
for article in response:

    openalex_id = article.get('id')
    
    authorships = article.get('authorships')

    for author in authorships:
        affiliations = author.get('affiliations')
        for affiliation in affiliations:
            #if 'https://openalex.org/I74656192' in affiliation.get('institution_ids'):
            if any(map(lambda id: id in ['https://openalex.org/I74656192', # GAU
                                         'https://openalex.org/I4387154063', # SUB
                                         'https://openalex.org/I4210091733', # GWDG
                                         'https://openalex.org/I4210116730' # UMG
                                        ], 
                       affiliation.get('institution_ids'))):
                author_name = author.get('author').get('display_name')
                raw_affiliation_string = affiliation.get('raw_affiliation_string')
                mapping_dict = map_address(raw_affiliation_string, gerit_mapping)

                faculty = mapping_dict.get('faculty')
                faculty_id = mapping_dict.get('faculty_id')

                department = mapping_dict.get('department')
                department_id = mapping_dict.get('department_id')

                institute = mapping_dict.get('institute')
                institute_id = mapping_dict.get('institute_id')

                print(f'OpenAlex Works ID: {openalex_id}')
                
                print(f'Person: {author_name}')
                print(f'Affiliationsstring: {raw_affiliation_string}')
                print(f'Fakultät: {faculty}')
                print(f'Fakultät ID: {faculty_id}')
                print(f'Department: {department}')
                print(f'Department ID: {department_id}')
                print(f'Institut: {institute}')
                print(f'Institut ID: {institute_id}')

                print('#############################')

OpenAlex Works ID: https://openalex.org/W4408636234
Person: Bertram Brenig
Affiliationsstring: Institute of Veterinary Medicine, University of Göttingen, Göttingen, Germany
Fakultät: Fakultät für Agrarwissenschaften
Fakultät ID: 13696
Department: Department Nutztierwissenschaften
Department ID: 157084871
Institut: Tierärztliches Institut
Institut ID: 16812
#############################
OpenAlex Works ID: https://openalex.org/W4408789761
Person: Judith Charlotte Witzel
Affiliationsstring: Department of Trauma, Orthopedics and Reconstructive Surgery, University Medical Center Göttingen 2 , ,
Fakultät: Universitätsmedizin Göttingen (UMG)
Fakultät ID: 13702
Department: Klinik für Unfallchirurgie, Orthopädie und Plastische Chirurgie
Department ID: 23145
Institut: None
Institut ID: None
#############################
OpenAlex Works ID: https://openalex.org/W4408789761
Person: Judith Charlotte Witzel
Affiliationsstring: J Witzel, Department of Trauma, Orthopedics and Reconstructive Surgery, Un